# Качество, тесты и ETL — 30 заданий

Практика на eBay; решений нет.

## Результаты обучения

После **Качество и ETL** вы должны объяснить transformation как plan, предсказать action/jobs/shuffle, связать schema/grain с результатом и доказать физическую эффективность через explain/UI/metrics.

## Ментальная модель исполнения

Data contract превращается в DataFrame assertions. Accepted/reject, run metadata, watermark и reconciliation формируют управляемый batch, а не просто write.

```text
transformations → unresolved logical plan
       ↓ analysis (catalog/types)
   optimized logical plan (Catalyst)
       ↓ physical planning / AQE
job → stage → shuffle → stage
       tasks             tasks
       └──── executors ─────┘
```
Action создаёт job. Один notebook/application может породить много jobs, а один job —
несколько stages. `repartition`, join и groupBy часто добавляют Exchange.

## Данные eBay

Grain eBay — `itemid` в `snapshot_dt`; 2 501 511 строк, 24 колонки, Parquet/Snappy.
Partition column — дата снимка. Цена, продавец, категории и доставка денормализованы.
Перед `latest item` или dedup проверяйте уникальность пары и задавайте tie-breaker.

Полная схема и проверки качества находятся в `data-catalog`. Raw read-only, результаты — в личном `spark_training`.

## Алгоритм решения

1. Зафиксируйте входной и целевой grain. 2. Выберите только нужные columns/rows. 3. Соберите transformation без action. 4. Проверьте schema и explain. 5. Предскажите partitions/shuffle. 6. Выполните минимальный action/write. 7. Повторно прочитайте и сверяйте keys/metrics. 8. Сохраните evidence.

Докажите schema, NULL/domain/key, read=accepted+rejected, target delta и rerun.

## Типичные ошибки

- Вызывать count/show после каждого шага и создавать лишние jobs.
- Использовать Python UDF при наличии встроенной функции.
- Делать repartition без понимания Exchange и целевого файла.
- Broadcast большой стороны или collect на driver.
- Кэшировать одноразовый DataFrame без materialization/unpersist.
- Измерять скорость при разных результатах или непрогретом JVM.

## Самопроверка

1. Какой action создаёт job? 2. Где появится shuffle? 3. Сколько input/output partitions? 4. Видит ли Catalyst выражение? 5. Каков grain после JOIN/window? 6. Как проверить idempotent rerun?

## Подробная теория

### 1. Контракт

Schema, grain, key, NULL, domain, range и freshness должны стать executable checks.

### 2. Reject

Отклонённая строка сохраняет reason, исходные поля и run_id.

### 3. Идемпотентность

Повтор порции приводит к тому же target через overwrite partition или устойчивый merge.

### 4. Инкремент

Watermark фиксируют после publish, late data требует overlap/reprocessing.

### 5. Тесты

Проверяйте schema, ключи, множества строк и метрики, а не только count.

## Сдача

Каждое задание записывает непустой Parquet в личный HDFS и evidence с transformation, observation и explanation. Checker использует активную SparkSession.

In [ ]:
import os,sys
sys.path.insert(0,'/opt/lab/spark-training')
from check_task import check_task,save_evidence
from pyspark.sql import SparkSession,functions as F,types as T,Window
spark=SparkSession.builder.appName('spark-training').enableHiveSupport().getOrCreate()
USER=os.environ.get('HDFS_USER',os.environ.get('HADOOP_USER_NAME','student'))
ROOT=f'hdfs://namenode:8020/user/{USER}/spark_training'
SOURCE='hdfs://namenode:8020/data/raw/ebay'
ebay=spark.read.parquet(SOURCE)
print('Spark',spark.version,'rows',ebay.count(),'columns',len(ebay.columns))

### Задание 1. schema contract

Создайте результат по теме **schema contract** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_01")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',1)

### Задание 2. required columns

Создайте результат по теме **required columns** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_02")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',2)

### Задание 3. type contract

Создайте результат по теме **type contract** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_03")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',3)

### Задание 4. not null

Создайте результат по теме **not null** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_04")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',4)

### Задание 5. domain check

Создайте результат по теме **domain check** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_05")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',5)

### Задание 6. range check

Создайте результат по теме **range check** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_06")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',6)

### Задание 7. unique key

Создайте результат по теме **unique key** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_07")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',7)

### Задание 8. duplicate profile

Создайте результат по теме **duplicate profile** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_08")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',8)

### Задание 9. reject frame

Создайте результат по теме **reject frame** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_09")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',9)

### Задание 10. accepted frame

Создайте результат по теме **accepted frame** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_10")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',10)

### Задание 11. row reconciliation

Создайте результат по теме **row reconciliation** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_11")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',11)

### Задание 12. metric reconciliation

Создайте результат по теме **metric reconciliation** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_12")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',12)

### Задание 13. assertions

Создайте результат по теме **assertions** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_13")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',13)

### Задание 14. dataframe equality

Создайте результат по теме **dataframe equality** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_14")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',14)

### Задание 15. deterministic sort

Создайте результат по теме **deterministic sort** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_15")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',15)

### Задание 16. idempotent overwrite

Создайте результат по теме **idempotent overwrite** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_16")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',16)

### Задание 17. incremental partition

Создайте результат по теме **incremental partition** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_17")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',17)

### Задание 18. watermark

Создайте результат по теме **watermark** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_18")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',18)

### Задание 19. late data

Создайте результат по теме **late data** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_19")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',19)

### Задание 20. dedup snapshot

Создайте результат по теме **dedup snapshot** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_20")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',20)

### Задание 21. latest record

Создайте результат по теме **latest record** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_21")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',21)

### Задание 22. SCD1

Создайте результат по теме **SCD1** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_22")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',22)

### Задание 23. SCD2

Создайте результат по теме **SCD2** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_23")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',23)

### Задание 24. audit columns

Создайте результат по теме **audit columns** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_24")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',24)

### Задание 25. run metadata

Создайте результат по теме **run metadata** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_25")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',25)

### Задание 26. quality report

Создайте результат по теме **quality report** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_26")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',26)

### Задание 27. publish gate

Создайте результат по теме **publish gate** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_27")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',27)

### Задание 28. rerun test

Создайте результат по теме **rerun test** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_28")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',28)

### Задание 29. failure recovery

Создайте результат по теме **failure recovery** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_29")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',29)

### Задание 30. ETL pipeline

Создайте результат по теме **ETL pipeline** и запишите `mode("overwrite").parquet(f"{ROOT}/quality_etl/task_30")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'quality_etl',30)